In [59]:
import pandas as pd 
import pyodbc
import sqlalchemy as sal
from sqlalchemy import text
from sqlalchemy import create_engine
import logging 
import os
import time 
logging.basicConfig(
    filename="logs/Hospital_Analysis.log",
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

In [60]:
df=pd.read_csv("appointments.csv")
df

,appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status
0,A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled
1,A002,P032,D004,2023-06-09,14:30:00,Therapy,No-show
2,A003,P048,D004,2023-06-28,8:00:00,Consultation,Cancelled
3,A004,P025,D006,2023-09-01,9:15:00,Consultation,Cancelled
4,A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show
...,...,...,...,...,...,...,...
195,A196,P045,D006,2023-10-26,9:45:00,Checkup,Cancelled
196,A197,P001,D005,2023-04-01,13:30:00,Emergency,No-show
197,A198,P022,D006,2023-05-15,8:30:00,Therapy,No-show
198,A199,P017,D001,2023-05-01,12:45:00,Follow-up,Completed


In [61]:
df.shape

(200, 7)

In [62]:
df.dtypes

appointment_id      str
patient_id          str
doctor_id           str
appointment_date    str
appointment_time    str
reason_for_visit    str
status              str
dtype: object

In [63]:
df.isnull().sum()

appointment_id      0
patient_id          0
doctor_id           0
appointment_date    0
appointment_time    0
reason_for_visit    0
status              0
dtype: int64

In [64]:
df.duplicated().sum()

np.int64(0)

In [65]:
df['appointment_date'] = pd.to_datetime(df['appointment_date'],errors='coerce')
df['appointment_time'] = pd.to_datetime(df['appointment_time'], format='%H:%M:%S',errors='coerce').dt.time

In [66]:
df.dtypes

appointment_id                 str
patient_id                     str
doctor_id                      str
appointment_date    datetime64[us]
appointment_time            object
reason_for_visit               str
status                         str
dtype: object

In [67]:
df.describe(include='all')

,appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status
count,200,200,200,200,200,200,200
unique,200,48,10,NaN,40,5,4
top,A001,P012,D005,NaN,15:30:00,Checkup,No-show
freq,1,10,29,NaN,9,45,52
mean,NaN,NaN,NaN,2023-06-17 11:38:24,NaN,NaN,NaN
min,NaN,NaN,NaN,2023-01-01 00:00:00,NaN,NaN,NaN
25%,NaN,NaN,NaN,2023-03-27 18:00:00,NaN,NaN,NaN
50%,NaN,NaN,NaN,2023-06-10 00:00:00,NaN,NaN,NaN
75%,NaN,NaN,NaN,2023-09-15 12:00:00,NaN,NaN,NaN
max,NaN,NaN,NaN,2023-12-30 00:00:00,NaN,NaN,NaN


In [68]:
appointment_df=df

In [69]:
df=pd.read_csv("billing.csv")

In [70]:
df

,bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status
0,B001,P034,T001,2023-08-09,3941.97,Insurance,Pending
1,B002,P032,T002,2023-06-09,4158.44,Insurance,Paid
2,B003,P048,T003,2023-06-28,3731.55,Insurance,Paid
3,B004,P025,T004,2023-09-01,4799.86,Insurance,Failed
4,B005,P040,T005,2023-07-06,582.05,Credit Card,Pending
...,...,...,...,...,...,...,...
195,B196,P045,T196,2023-10-26,2477.80,Cash,Pending
196,B197,P001,T197,2023-04-01,975.49,Cash,Pending
197,B198,P022,T198,2023-05-15,3383.72,Cash,Failed
198,B199,P017,T199,2023-05-01,1472.17,Credit Card,Paid


In [71]:
df.shape

(200, 7)

In [72]:
df.dtypes

bill_id               str
patient_id            str
treatment_id          str
bill_date             str
amount            float64
payment_method        str
payment_status        str
dtype: object

In [73]:
df.isnull().sum()

bill_id           0
patient_id        0
treatment_id      0
bill_date         0
amount            0
payment_method    0
payment_status    0
dtype: int64

In [74]:
df.duplicated().sum()

np.int64(0)

In [75]:
df['bill_date'] = pd.to_datetime(df['bill_date'],errors='coerce')

In [76]:
df.dtypes

bill_id                      str
patient_id                   str
treatment_id                 str
bill_date         datetime64[us]
amount                   float64
payment_method               str
payment_status               str
dtype: object

In [77]:
billing_df=df

In [78]:
df=pd.read_csv("doctors.csv")
df

,doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email
0,D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com
1,D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com
2,D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com
3,D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com
4,D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com
5,D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com
6,D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com
7,D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com
8,D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com
9,D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com


In [79]:
df.shape

(10, 8)

In [80]:
df.dtypes

doctor_id             str
first_name            str
last_name             str
specialization        str
phone_number        int64
years_experience    int64
hospital_branch       str
email                 str
dtype: object

In [81]:
doctors_df=df

In [82]:
doctors_df.isnull().sum()

doctor_id           0
first_name          0
last_name           0
specialization      0
phone_number        0
years_experience    0
hospital_branch     0
email               0
dtype: int64

In [83]:
doctors_df.duplicated().sum()

np.int64(0)

In [84]:
df=pd.read_csv("patients.csv")
df

,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
0,P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
1,P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
2,P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
3,P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
4,P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com
5,P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com
6,P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com
7,P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com
8,P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com
9,P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com


In [85]:
df.shape

(50, 11)

In [86]:
df.dtypes

patient_id              str
first_name              str
last_name               str
gender                  str
date_of_birth           str
contact_number        int64
address                 str
registration_date       str
insurance_provider      str
insurance_number        str
email                   str
dtype: object

In [87]:
df.isnull().sum()

patient_id            0
first_name            0
last_name             0
gender                0
date_of_birth         0
contact_number        0
address               0
registration_date     0
insurance_provider    0
insurance_number      0
email                 0
dtype: int64

In [88]:
df.duplicated().sum()

np.int64(0)

,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
0,P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
1,P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
2,P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
3,P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
4,P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com
5,P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com
6,P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com
7,P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com
8,P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com
9,P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com


In [89]:
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'],errors='coerce')
df['registration_date'] = pd.to_datetime(df['registration_date'],errors='coerce')

In [90]:
df.dtypes

patient_id                       str
first_name                       str
last_name                        str
gender                           str
date_of_birth         datetime64[us]
contact_number                 int64
address                          str
registration_date     datetime64[us]
insurance_provider               str
insurance_number                 str
email                            str
dtype: object

In [91]:
patients_df=df

In [92]:
df=pd.read_csv("treatments.csv")
df

,treatment_id,appointment_id,treatment_type,description,cost,treatment_date
0,T001,A001,Chemotherapy,Basic screening,3941.97,2023-08-09
1,T002,A002,MRI,Advanced protocol,4158.44,2023-06-09
2,T003,A003,MRI,Standard procedure,3731.55,2023-06-28
3,T004,A004,MRI,Basic screening,4799.86,2023-09-01
4,T005,A005,ECG,Standard procedure,582.05,2023-07-06
...,...,...,...,...,...,...
195,T196,A196,Chemotherapy,Advanced protocol,2477.80,2023-10-26
196,T197,A197,Physiotherapy,Standard procedure,975.49,2023-04-01
197,T198,A198,ECG,Basic screening,3383.72,2023-05-15
198,T199,A199,Chemotherapy,Basic screening,1472.17,2023-05-01


In [93]:
df.shape

(200, 6)

In [94]:
df.dtypes

treatment_id          str
appointment_id        str
treatment_type        str
description           str
cost              float64
treatment_date        str
dtype: object

In [95]:
df.isnull().sum()

treatment_id      0
appointment_id    0
treatment_type    0
description       0
cost              0
treatment_date    0
dtype: int64

In [96]:
df.duplicated().sum()

np.int64(0)

In [97]:
df['treatment_date'] = pd.to_datetime(df['treatment_date'],errors='coerce')

In [98]:
df.dtypes

treatment_id                 str
appointment_id               str
treatment_type               str
description                  str
cost                     float64
treatment_date    datetime64[us]
dtype: object

In [99]:
treatments_df=df

In [100]:
import urllib

In [103]:
engine = sal.create_engine(r'mssql://ABHISHEK\SQLEXPRESS/Hospital_db?driver=ODBC+DRIVER+17+FOR+SQL+SERVER')
conn=engine.connect() # For connnecting your SQL Database with Python with Hospital_db database

In [110]:
patients_df.to_sql(
    name='patients',      # table name in SQL Server
    con=engine,
    schema='dbo',                 # optional, defaults to dbo
    if_exists='replace',          # 'replace', 'append', or 'fail'
    index=False,                  # don't write the pandas index as a column
    chunksize=1000,               # helps with large dataframes
    method='multi'                # faster batch inserts
)

50

In [105]:
len(patients_df) # appointment_df, billing_df, doctors_df, treatments_df

50

In [111]:
appointment_df.to_sql(
    name='appointment',      # table name in SQL Server
    con=engine,
    schema='dbo',                 # optional, defaults to dbo
    if_exists='replace',          # 'replace', 'append', or 'fail'
    index=False,                  # don't write the pandas index as a column
    chunksize=1000,               # helps with large dataframes
    method='multi'                # faster batch inserts
)

200

In [112]:
billing_df.to_sql(
    name='billing',      # table name in SQL Server
    con=engine,
    schema='dbo',                 # optional, defaults to dbo
    if_exists='replace',          # 'replace', 'append', or 'fail'
    index=False,                  # don't write the pandas index as a column
    chunksize=1000,               # helps with large dataframes
    method='multi'                # faster batch inserts
)

200

In [113]:
doctors_df.to_sql(
    name='doctors',      # table name in SQL Server
    con=engine,
    schema='dbo',                 # optional, defaults to dbo
    if_exists='replace',          # 'replace', 'append', or 'fail'
    index=False,                  # don't write the pandas index as a column
    chunksize=1000,               # helps with large dataframes
    method='multi'                # faster batch inserts
)

10

In [114]:
treatments_df.to_sql(
    name='treatments',      # table name in SQL Server
    con=engine,
    schema='dbo',                 # optional, defaults to dbo
    if_exists='replace',          # 'replace', 'append', or 'fail'
    index=False,                  # don't write the pandas index as a column
    chunksize=1000,               # helps with large dataframes
    method='multi'                # faster batch inserts
)

200